In [97]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
from IPython.display import display, HTML
import sqlite3
import mysql.connector

In [2]:
import pyarrow

### Extract

In [3]:
data = yf.download("BTC-USD"
                   ,start = "2023-01-01"
                   ,end = "2023-12-16")

print(data)

[*********************100%%**********************]  1 of 1 completed

                    Open          High           Low         Close  \
Date                                                                 
2023-01-01  16547.914062  16630.439453  16521.234375  16625.080078   
2023-01-02  16625.509766  16759.343750  16572.228516  16688.470703   
2023-01-03  16688.847656  16760.447266  16622.371094  16679.857422   
2023-01-04  16680.205078  16964.585938  16667.763672  16863.238281   
2023-01-05  16863.472656  16884.021484  16790.283203  16836.736328   
...                  ...           ...           ...           ...   
2023-12-11  43792.019531  43808.375000  40234.578125  41243.832031   
2023-12-12  41238.734375  42048.304688  40667.562500  41450.222656   
2023-12-13  41468.464844  43429.781250  40676.867188  42890.742188   
2023-12-14  42884.261719  43390.859375  41767.089844  43023.972656   
2023-12-15  43028.250000  43087.824219  41692.968750  41929.757812   

               Adj Close       Volume  
Date                                   
2023-01-0

In [89]:
exchange = "BTC-USD"
get_date = datetime.now()
get_date_formated = get_date.strftime("%Y-%m-%d")
get_date_730_days_ago = get_date - timedelta(days=729)
get_date_730_days_ago_formated = get_date_730_days_ago.strftime("%Y-%m-%d")

daily_data = yf.download(exchange
                 ,start = "2012-01-01"
                 ,end = get_date_formated)



hour_data = yf.download("BTC-USD"
                 ,start = get_date_730_days_ago_formated
                 ,end = get_date_formated
                 ,interval = "1h")





[*********************100%%**********************]  1 of 1 completed


[*********************100%%**********************]  1 of 1 completed


### Transform

#### Daily_data

In [90]:
# Add the exchange column
daily_data['exchange'] = exchange

#Reset index to extract date column
daily_data.reset_index(inplace=True)

#Extract the date part of the datetime column
# We are converting the Date to id_date format
# Example: 2024-04-17 is converted to 20240417

daily_data['Date'] = daily_data['Date'].astype(str)
daily_data['Date'] = daily_data['Date'].str.replace('-', '')

#### Hour_data:

In [91]:
# Add the exchange column
hour_data['exchange'] = exchange

# Reset index to extract date column
hour_data.reset_index(inplace=True)

# Separe the date and time parts
hour_data['Datetime'] = hour_data['Datetime'].astype(str)
hour_data[['Date', 'Hour']] = hour_data['Datetime'].str.split(' ', expand=True)

# Extract date and hour data
# We are converting the Date to id_date format
# Example: 2024-04-17 is converted to 20240417

hour_data['Date'] = hour_data['Date'].str.replace('-', '')
hour_data['Hour'] = hour_data['Hour'].str.split(':', expand=True)[0]
hour_data['Hour'] = hour_data['Hour'].astype(int)

#Drop datetime data
hour_data = hour_data.drop(columns = ['Datetime'])

#### DT_Exchanges to add id_exchange:

We extract the DT_Exchanges to add the id_exchange to the dataframes

In [92]:
import pandas as pd
import sqlite3

connection = mysql.connector.connect(
    user = 'root',
    password = 'root',
    host = 'localhost',
    port = 3306,
    database = 'Historical_Data'
)
print("MySQL DB Connected")

cursor = connection.cursor()

cursor.execute("SELECT * FROM DT_EXCHANGES")

results = cursor.fetchall()


columns = [column[0] for column in cursor.description]


df_dt_exchanges = pd.DataFrame(results, columns=columns)


cursor.close()
connection.close()

df_dt_exchanges = df_dt_exchanges[['exchange', 'id_exchange']]
df_dt_exchanges.head(2)


MySQL DB Connected


,exchange,id_exchange
0,BTC-USD,49
1,ETH-USD,50


In [93]:
daily_data = pd.merge(daily_data, df_dt_exchanges,
                      on='exchange', how='left')

hour_data = pd.merge(hour_data, df_dt_exchanges,
                     on = 'exchange', how = 'left')

### Load

### MYSQL

#### Daily_data

In [100]:
connection = mysql.connector.connect(
    user = 'root',
    password = 'root',
    host = 'localhost',
    port = 3306,
    database = 'Historical_Data'
)
print("MySQL DB Connected")

cursor = connection.cursor()

cursor.execute("""SET FOREIGN_KEY_CHECKS = 0""")


# SQL Consult
sql_insert = """
    INSERT INTO FT_DAILY_DATA (id_date, Open, High, Low, Close, adj_close, Volume, Exchange, id_exchange)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

try:
    # Iterate on the dataframe
    for index, row in daily_data.iterrows():
        cursor.execute(sql_insert, tuple(row))
    
    # Confirm the changes on the database
    connection.commit()
    print("Data inserted correctly on the DAILY_DATA table.")
except mysql.connector.Error as error:
    # Error
    print("Error inserting the DAILY data:", error)
    connection.rollback()
    
cursor.execute("""SET FOREIGN_KEY_CHECKS = 1""")

# Close the cursor and the connection
cursor.close()
connection.close()

MySQL DB Connected
Data inserted correctly on the DAILY_DATA.


#### Hour_data

In [104]:
hour_data.columns

Index(['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'exchange',
       'Date', 'Hour', 'id_exchange'],
      dtype='object')

In [106]:
connection = mysql.connector.connect(
    user = 'root',
    password = 'root',
    host = 'localhost',
    port = 3306,
    database = 'Historical_Data'
)
print("MySQL DB Connected")

cursor = connection.cursor()

cursor.execute("""SET FOREIGN_KEY_CHECKS = 0""")


# SQL Consult
sql_insert = """
    INSERT INTO FT_HOUR_DATA (Open, High, Low, Close, adj_close, Volume, Exchange, id_date, Hour, id_exchange)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

try:
    # Iterate on the dataframe
    for index, row in hour_data.iterrows():
        cursor.execute(sql_insert, tuple(row))
    
    # Confirm the changes on the database
    connection.commit()
    print("Data inserted correctly on the HOUR_DATA table.")
except mysql.connector.Error as error:
    # Error
    print("Error inserting the HOUR data:", error)
    connection.rollback()
    
cursor.execute("""SET FOREIGN_KEY_CHECKS = 1""")

# Close the cursor and the connection
cursor.close()
connection.close()

MySQL DB Connected
Data inserted correctly on the HOUR_DATA table.


### Save dataframes into Parquet files

In [11]:
#daily_data.to_parquet('../data/BTC-USD/daily_data')

In [12]:
#hour_data.to_parquet('../data/BTC-USD/hour_data')